# Queries de verificación

Consultas analíticas sobre el modelo implementado en BigQuery, para comprobar que
los datos cargados tienen sentido de negocio y que el modelo soporta las
necesidades de análisis planteadas en el enunciado.

In [5]:
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv

load_dotenv()

credentials = service_account.Credentials.from_service_account_file(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
)
client = bigquery.Client(project=os.getenv("GCP_PROJECT_ID"), credentials=credentials)
dataset_id = f"{client.project}.{os.getenv('BQ_DATASET_ID')}"
print("Conectado a:", dataset_id)

Conectado a: tc-sql-maria-muriel.techmuriel


## 1. Ingresos por mes

Suma de pagos completados agrupados por mes, para ver la evolución de ingresos reales
(excluye pagos fallidos o reembolsados).

In [6]:
query = f"""
SELECT
  FORMAT_DATE('%Y-%m', payment_date) AS mes,
  ROUND(SUM(amount), 2) AS ingresos
FROM `{dataset_id}.payments`
WHERE status = 'completed'
GROUP BY mes
ORDER BY mes;
"""
client.query(query).to_dataframe()

c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,mes,ingresos
0,2025-09,306067.34
1,2025-10,351085.13
2,2025-11,311219.25
3,2025-12,275909.94
4,2026-01,314185.96
5,2026-02,314413.36
6,2026-03,383397.42
7,2026-04,349827.46
8,2026-05,382921.45
9,2026-06,363232.34


## 2. Top 10 productos más vendidos

Por unidades vendidas, uniendo `order_items` con `products` para mostrar el nombre.

In [7]:
query = f"""
SELECT
  p.name,
  SUM(oi.quantity) AS unidades_vendidas
FROM `{dataset_id}.order_items` oi
JOIN `{dataset_id}.products` p ON oi.product_id = p.product_id
GROUP BY p.name
ORDER BY unidades_vendidas DESC
LIMIT 10;
"""
client.query(query).to_dataframe()

c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,name,unidades_vendidas
0,Extended demand-driven moratorium,184
1,Robust 24/7 productivity,172
2,Future-proofed zero-defect Internet solution,170
3,Configurable bifurcated support,162
4,Upgradable hybrid knowledge user,157
5,Managed grid-enabled Internet solution,156
6,Customizable fault-tolerant knowledge user,155
7,Optional zero-defect standardization,153
8,Exclusive systemic hub,152
9,Enterprise-wide dedicated middleware,151


## 3. Clientes por país

Distribución geográfica de la base de clientes.

In [8]:
query = f"""
SELECT
  country,
  COUNT(*) AS total_clientes
FROM `{dataset_id}.customers`
GROUP BY country
ORDER BY total_clientes DESC;
"""
client.query(query).to_dataframe()

c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,country,total_clientes
0,France,93
1,Spain,93
2,Netherlands,88
3,Portugal,83
4,Germany,75
5,Italy,68


## 4. Tiempo medio de entrega

Días entre `order_date` y `delivered_date`, solo para pedidos ya entregados.

In [9]:
query = f"""
SELECT
  ROUND(AVG(DATE_DIFF(delivered_date, order_date, DAY)), 1) AS dias_medios_entrega
FROM `{dataset_id}.orders`
WHERE status = 'delivered';
"""
client.query(query).to_dataframe()

c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,dias_medios_entrega
0,7.1


## 5. Valoración media por categoría

Rating medio de las reviews, uniendo `reviews` → `order_items` → `products` → `categories`.

In [10]:
query = f"""
SELECT
  c.name AS categoria,
  ROUND(AVG(r.rating), 2) AS rating_medio,
  COUNT(r.review_id) AS total_reviews
FROM `{dataset_id}.reviews` r
JOIN `{dataset_id}.order_items` oi ON r.order_item_id = oi.order_item_id
JOIN `{dataset_id}.products` p ON oi.product_id = p.product_id
JOIN `{dataset_id}.categories` c ON p.category_id = c.category_id
GROUP BY categoria
ORDER BY rating_medio DESC;
"""
client.query(query).to_dataframe()

c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,categoria,rating_medio,total_reviews
0,Smartphones,4.23,94
1,Tablets,4.16,96
2,Laptops,4.16,163
3,Audio,4.13,88
4,Componentes,4.12,200
5,Accesorios,4.12,109
6,Periféricos,4.11,157
7,Wearables,3.94,139


## 6. Margen (bonus) — ingresos vs coste por categoría

In [11]:
query = f"""
SELECT
  c.name AS categoria,
  ROUND(SUM(oi.quantity * oi.unit_price - oi.discount), 2) AS ingresos,
  ROUND(SUM(oi.quantity * p.cost), 2) AS coste_total,
  ROUND(SUM(oi.quantity * oi.unit_price - oi.discount) - SUM(oi.quantity * p.cost), 2) AS margen
FROM `{dataset_id}.order_items` oi
JOIN `{dataset_id}.products` p ON oi.product_id = p.product_id
JOIN `{dataset_id}.categories` c ON p.category_id = c.category_id
GROUP BY categoria
ORDER BY margen DESC;
"""
client.query(query).to_dataframe()

c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,categoria,ingresos,coste_total,margen
0,Laptops,859135.39,628136.17,230999.22
1,Componentes,732490.66,546561.48,185929.18
2,Wearables,688821.96,513668.14,175153.82
3,Periféricos,642125.77,490570.50,151555.27
4,Accesorios,517868.77,390265.46,127603.31
5,Audio,472568.67,356206.39,116362.28
6,Smartphones,322385.53,216634.22,105751.31
7,Tablets,494236.41,407578.09,86658.32
